# JAX: Dense GEMMs with TransformerEngine

This notebook walks through replacing a plain `flax.linen.Dense`'s GEMM with TransformerEngine's quantized GEMM. We start with the headline single-GPU speedup on a single Dense layer, look at the numerical cost, then scale the same Dense layer to a DP=2/TP=2 mesh.

**Audience.** You're comfortable with Flax Linen: `nn.Module`, `init`/`apply`, variable collections, and RNG handling. We won't re-introduce those.

**Recipe.** We use `MXFP8BlockScaling` as the default — it has no extra runtime state to manage. MXFP8 requires a Blackwell-class GPU; on Hopper, swap in `DelayedScaling` or `Float8CurrentScaling`.

[← Back to the JAX integration overview](../te_jax_integration.ipynb)

## 1. Baseline: a plain Flax Dense block

We isolate the optimization to a single linear layer so it's clear what's changing. `dot_general_cls` is exposed as a constructor argument so we can swap in TE later without touching the model definition.

In [1]:
import sys
sys.path.append("..")  # so we can import quickstart_jax_utils from docs/examples/

import warnings
# Silence a benign TE warning about tpsp_resource; tp_resource and tpsp_resource
# are mutually exclusive in TE's MeshResource, and tp_resource is what we want.
warnings.filterwarnings("ignore", message="Tensor sequence parallelism is detected")

import jax
import jax.numpy as jnp
from flax import linen as nn
import quickstart_jax_utils as utils

In [2]:
class FlaxDenseBlock(nn.Module):
    """One linear layer. `dot_general_cls` lets us swap the GEMM impl."""
    features: int
    dot_general_cls: callable = lambda: None

    @nn.compact
    def __call__(self, x):
        return nn.Dense(
            features=self.features,
            use_bias=False,
            dot_general=self.dot_general_cls(),
        )(x)

# Shapes chosen to be large enough that quantization actually pays off.
batch, seq, hidden, out_features = 4, 2048, 4096, 16384
dtype = jnp.bfloat16

key = jax.random.PRNGKey(0)
k_init, k_x, k_dy = jax.random.split(key, 3)
x = jax.random.normal(k_x, (batch, seq, hidden)).astype(dtype)
dy = jax.random.normal(k_dy, (batch, seq, out_features)).astype(dtype)

baseline = FlaxDenseBlock(features=out_features)
baseline_vars = baseline.init(k_init, x)

## 2. Quantized Dense via `make_dot_general_cls`

TE exposes a single helper, `te_flax.make_dot_general_cls(recipe)`, that returns a Flax module class you pass directly to `nn.Dense(..., dot_general=...)`. Internally it routes the GEMM through `transformer_engine.jax.dense.dense`, which handles the cast, the quantized matmul, and the VJP.

Crucially, **your model parameters are still yours**. TE doesn't create the `kernel`; it only wraps the multiply. Initialization, sharding annotations, and optimizer state stay where they were.

In [3]:
from transformer_engine.jax import flax as te_flax
from transformer_engine.common.recipe import MXFP8BlockScaling

recipe = MXFP8BlockScaling()
te_dot_general_cls = te_flax.make_dot_general_cls(recipe)

te_model = FlaxDenseBlock(features=out_features, dot_general_cls=te_dot_general_cls)
te_vars = te_model.init(k_init, x)

print("Variable collections:", list(te_vars.keys()))
print(jax.tree_util.tree_map(lambda a: (a.shape, a.dtype), te_vars))

Variable collections: ['params']
{'params': {'Dense_0': {'kernel': ((4096, 16384), dtype('float32'))}}}


## 3. Does it actually help? — single-GPU performance

Let's measure first, ask questions later. `speedometer` runs a JIT-compiled forward+backward loop with warmup, on the same input for both models.

In [4]:
print("bf16 baseline:")
utils.speedometer(
    model_apply_fn=baseline.apply,
    variables=baseline_vars,
    input=x, output_grad=dy,
)

print(f"\nTE {type(recipe).__name__}:")
utils.speedometer(
    model_apply_fn=te_model.apply,
    variables=te_vars,
    input=x, output_grad=dy,
)

bf16 baseline:


Mean time: 4.13301944732666 ms

TE MXFP8BlockScaling:


E0511 13:59:25.678780  152730 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


Mean time: 1.6977357864379883 ms


On a single GB200, that's roughly **2.5× faster** for the fwd+bwd of one large Dense — and the only code change was passing `dot_general=te_dot_general_cls()` into `nn.Dense`.

The speedup depends on shape: large GEMMs benefit most. Very small GEMMs may not benefit at all because the cast + scale overhead can dominate.

<div class="alert alert-warning">
<b>Remat / activation checkpointing.</b> If your training loop uses <code>jax.checkpoint_policies.checkpoint_dots</code> (or any policy that matches <code>jax.lax.dot_general</code>), swap it for <code>transformer_engine.jax.checkpoint_policies.checkpoint_dots_and_te_gemms</code>. Otherwise TE's quantized GEMM primitives won't be checkpointed correctly and your performance comparison will be unfair.
</div>

## 4. What does it cost? — numerical comparison

Quantization is lossy by construction. `compare_fwd_bwd` in `quickstart_jax_utils` runs both apply functions through `jax.value_and_grad` on the same input and reports max abs/rel diff on the output, the input gradient, and the kernel gradient.

In [5]:
# Make the TE model's kernel identical to the baseline's so we measure quantization
# error specifically, not init differences.
te_vars_matched = {**te_vars, "params": baseline_vars["params"]}

diffs = utils.compare_fwd_bwd(
    apply_a=baseline.apply, variables_a=baseline_vars,
    apply_b=te_model.apply, variables_b=te_vars_matched,
    input=x, output_grad=dy,
)
for name, d in diffs.items():
    print(f"{name}: max_abs={d['max_abs']:.3e}  max_rel={d['max_rel']:.3e}")

y: max_abs=2.245e-01  max_rel=3.881e-02
dx: max_abs=4.766e-01  max_rel=4.144e-02
dW: max_abs=1.874e+01  max_rel=3.676e-02


Max relative error around a few percent is the expected MXFP8 fidelity envelope. In real training, gradient noise dominates this error.

## 5. A note on recipe state

MXFP8 is stateless — scaling factors are computed from each tensor as it flows through the GEMM, so there is nothing to persist across steps. If you swap in `DelayedScaling` instead, `init` will produce a second variable collection, `_overwrite_with_gradient`, holding `kernel_amax_history`, `kernel_scale`, `x_amax_history`, `x_scale`, etc. These are **not** model parameters — they are Flax variables that TE updates each step to compute per-tensor scales from a rolling amax window.

If you use a stateful recipe, you must thread the *entire* `var_collect` through your training loop (not just `params`) so the history persists across steps. `MXFP8BlockScaling` avoids this entirely by scaling locally per 32-element block.

## 6. Multi-GPU: DP=2 / TP=2 on a single Dense

**Prerequisite:** this section requires four GPUs.

Keeping the same `FlaxDenseBlock` from the rest of the notebook, we run it on a 2×2 mesh with **data parallelism** on one axis and **tensor parallelism** (column-parallel: shard the kernel's output dim) on the other.

Two pieces wire this up:

1. A `jax.sharding.Mesh` you build once at module scope (outside JIT).
2. TE's `MeshResource`, set globally via `global_shard_guard`, which tells TE which mesh axes are DP and TP.

In [6]:
n_devices = len(jax.devices())
print(f"Visible devices: {n_devices}")
assert n_devices >= 4, "This section requires 4 GPUs for DP=2/TP=2."

Visible devices: 4


In [7]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from jax.experimental import mesh_utils
from transformer_engine.jax.sharding import MeshResource, global_shard_guard

# 2x2 mesh: DP on one axis, TP on the other.
devices = mesh_utils.create_device_mesh((2, 2))
mesh = Mesh(devices, axis_names=("dp", "tp"))

# Tell TE which mesh axis is which. This is a *global* setting, established
# outside JIT, so TE's GEMM primitives can plan comms accordingly.
mesh_resource = MeshResource(dp_resource="dp", tp_resource="tp")

**Sharding plan:**

| Tensor | Shape | PartitionSpec |
|---|---|---|
| Kernel (column-parallel) | `(hidden, out_features)` | `P(None, "tp")` |
| Input activations | `(batch, seq, hidden)` | `P("dp", None, None)` |
| Gradient on output | `(batch, seq, out_features)` | `P("dp", None, "tp")` |

In [8]:
kernel_sharding = NamedSharding(mesh, P(None, "tp"))
input_sharding = NamedSharding(mesh, P("dp", None, None))
output_grad_sharding = NamedSharding(mesh, P("dp", None, "tp"))

def shard_kernel(variables):
    params = variables["params"]
    sharded = jax.device_put(params["Dense_0"]["kernel"], kernel_sharding)
    return {**variables, "params": {**params,
                                    "Dense_0": {**params["Dense_0"], "kernel": sharded}}}

x_mp_s = jax.device_put(x, input_sharding)
dy_mp_s = jax.device_put(dy, output_grad_sharding)
baseline_vars_s = shard_kernel(baseline_vars)
te_vars_s = shard_kernel(te_vars)

In [9]:
with jax.set_mesh(mesh), global_shard_guard(mesh_resource):
    print("bf16 DP=2/TP=2:")
    utils.speedometer(
        model_apply_fn=baseline.apply,
        variables=baseline_vars_s,
        input=x_mp_s, output_grad=dy_mp_s,
    )

    print(f"\nTE {type(recipe).__name__} DP=2/TP=2:")
    utils.speedometer(
        model_apply_fn=te_model.apply,
        variables=te_vars_s,
        input=x_mp_s, output_grad=dy_mp_s,
    )

bf16 DP=2/TP=2:


Mean time: 1.7351531982421875 ms

TE MXFP8BlockScaling DP=2/TP=2:


Mean time: 0.9787273406982422 ms


**Why this works.** Column-parallel TP shards the kernel along its *output* dim. The input activations are replicated along the TP axis, so the GEMM needs no input all-gather; it just runs locally on each TP shard and produces an output that's already sharded along the TP axis. The forward has no TP comms. The backward needs an all-reduce on the input-gradient (`dx`), which is partly hidden under the next layer's compute in a real model.

With TE, the bf16→FP8 cast and the GEMM both run on the local shard. Since the column-parallel pattern keeps comms out of the forward critical path, the GEMM-compute speedup translates directly to wall-clock.

The DP axis just shards the batch — it adds no comms in the forward; the gradient all-reduce on the backward is what gets DP-ified.

<div class="alert alert-info">
<b>Flax logical axes.</b> In larger models, prefer Flax's logical axis rules (set via <code>flax.linen.logical_axis_rules</code>) and annotate kernels with <code>kernel_axes=(...)</code> on TE modules. The manual approach above keeps this notebook small; the logical-axes approach scales to many layers without per-layer plumbing.
</div>

## 7. Collective GEMM (placeholder)

*Coming soon.*

A further optimization is to *overlap* the TP all-gather / all-reduce with the GEMM itself — a "collective GEMM". TE's JAX dense path exposes this via the `collective_op_set` argument on `transformer_engine.jax.dense.dense`. A future revision of this notebook will walk through:

- enabling overlapped AG+GEMM and reduce-scatter+GEMM,
- the throughput delta at large TP world sizes,
- when the overlap pays off and when it doesn't (small layers, slow interconnects).

## Recap

- One line of code (`dot_general=te_flax.make_dot_general_cls(recipe)()`) swaps in a quantized GEMM.
- Your `params` tree is unchanged; stateful recipes add a `_overwrite_with_gradient` collection that must be threaded through training.
- Numerical error from MXFP8 is small enough to be dominated by gradient noise in real training.
- On a single GB200, one large Dense fwd+bwd is **~2.5× faster** with MXFP8.
- The same Dense under DP=2/TP=2 stays faster with FP8 — column-parallel TP keeps comms out of the forward critical path, so the GEMM-compute win carries through.

**Next:** [Attention](./attention.ipynb) · [Mixture of Experts](./moe.ipynb) · [← Hub](../te_jax_integration.ipynb)